# Session 13：從表格到圖表

合成資料；一行＝一家店一年。金額單位：萬元（人民幣，練習設定）。

先猜 → 跑一格 → 說明 → 驗算。前六個動作是引導示範；最後是自己改題。
請把整個 session13 資料夾下載齊。可從倉庫根目錄或 session13 目錄執行。
故意刪格／重啟的狀態實驗另開 scratch notebook，依 live-runbook 操作。

第一次不要直接 Run All。完成挑戰後再保存、Restart Kernel、Run All，檢查能否乾淨重現。

## 0. 環境與路徑

選 Python kernel。先找得到檔案再繼續；以下不會安裝、下載或上傳任何東西。
缺套件時先確認現在選的 kernel；終端機的 Python 未必是同一個。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python:", sys.executable)
print("pandas:", pd.__version__, "matplotlib:", matplotlib.__version__)
print("Working directory:", Path.cwd())
lesson_dir = Path.cwd()
if not (lesson_dir / "data/sample-sales.csv").is_file():
    lesson_dir = lesson_dir / "session13"
csv_path = lesson_dir / "data/sample-sales.csv"
if not csv_path.is_file():
    raise FileNotFoundError("找不到 data/sample-sales.csv；請從 session13 或倉庫根目錄開啟。")
output_dir = lesson_dir / "outputs"
output_dir.mkdir(exist_ok=True)
print("CSV:", csv_path.resolve())
print("Outputs:", output_dir.resolve())

## 1. 讀：一行代表什麼？

先預測幾行幾欄，再執行。`category` 是主力品類標記，不是各品類的金額明細。

In [ ]:
df = pd.read_csv(csv_path)
print("shape:", df.shape)
print(df.head().to_string(index=False))

## 2. 看：不只看前五行

型別、缺值、店年是否重複？本樣本都正常，不表示別的資料也一樣。先檢查，再決定如何處理。

In [ ]:
print(df.dtypes)
print("Missing values:")
print(df.isna().sum())
print("Duplicate store/year keys:", df.duplicated(["store", "year"]).sum())
assert df.shape == (20, 5), "資料形狀與本課樣本不符"
assert not df.isna().any().any(), "先釐清缺值，不直接填 0"
assert not df.duplicated(["store", "year"]).any(), "先釐清重複鍵與資料粒度"

## 3. 選：一欄 vs 一欄的表

猜猜下面兩者的 shape 為何不同。

In [ ]:
profit_column = df["profit"]
profit_table = df[["profit"]]
print(type(profit_column).__name__, profit_column.shape)
print(type(profit_table).__name__, profit_table.shape)

## 4. 篩：利潤大於 20 的行

先用 CSV 手數，再跑。原始 df 保留，用 filtered 接新結果。
口頭挑戰：改成大於 30，哪些行會留下？先說預測再改。完成後還原成 20 再全跑。

In [ ]:
filtered = df[df["profit"] > 20]
print(filtered[["store", "year", "profit"]].to_string(index=False))
print("Rows after filtering:", len(filtered))
print("Original rows:", len(df))

## 5. 分組：各店五年利潤總和

先說會有幾組，年份會不會保留下來。手算 A 的 18＋22＋25＋30＋35，再抽查另一店。
最後核對分組合計和原表合計。只驗對一格還不夠。

In [ ]:
totals = df.groupby("store")["profit"].sum()
print(totals)
print("Grouped total:", totals.sum(), "Original total:", df["profit"].sum())
assert totals.sum() == df["profit"].sum()

## 6. 畫：已核對的表 → 五年比較圖

比較用條形圖。這是五年總和，不能當逐年趨勢。
下面字型設定只需課前確認：有常見中文字型就用中文，沒有就用英文。
英文 Store A–D 對應原表門店 A–D；10,000 CNY 就是萬元人民幣。

In [ ]:
from matplotlib import font_manager

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
cjk_font = next((name for name in ["Noto Sans CJK TC", "Noto Sans CJK SC", "Microsoft YaHei", "PingFang TC", "SimHei"] if name in available_fonts), None)
use_chinese = cjk_font is not None
if use_chinese:
    plt.rcParams["font.family"] = cjk_font
plt.rcParams["axes.unicode_minus"] = False
store_labels = {f"門店{letter}": f"Store {letter}" for letter in "ABCD"}
y_label = "利潤（萬元人民幣）" if use_chinese else "Profit (10,000 CNY)"
x_label = "門店" if use_chinese else "Store"
print("Chart font:", cjk_font or "English fallback")

In [ ]:
plot_totals = totals if use_chinese else totals.rename(index=store_labels)
ax = plot_totals.plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_title("各店 2021–2025 利潤合計（合成資料）" if use_chinese else "Profit by store, 2021–2025 total (synthetic)")
ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
fig = ax.get_figure()
fig.tight_layout()
bar_path = output_dir / "profit-total.png"
fig.savefig(bar_path, dpi=150, bbox_inches="tight")
print("Saved:", bar_path.resolve())
plt.show()
plt.close(fig)

## 7. 自己改題：2025 年各店比較（必做）

先用自己的話寫：問題、預期行數、單位、選圖理由。
在下面空白代碼格完成：保留原 df → 篩 2025 → 印出各店利潤表 → 畫條形圖 → 存 outputs/profit-2025.png。
圖要注明年份、合成資料與單位。先自己試，卡住時只向 AI 要下一步提示。

這是學生練習格，留白時不會產出該圖；不能把範例總和圖當成已完成挑戰。

In [ ]:
# 在這裡完成 2025 年比較；先預測，再執行。


## 8. 五年趨勢（進階，有時間才做）

保留 year 和 store，檢查店年唯一後用 pivot 把年份放 index、門店放 columns、profit 放 values。
年份按升序，四條線每條五個點，輸出 outputs/profit-trend.png。
為什麼前面的四個 totals 不夠？B 店是不是每年都成長？

In [ ]:
# 有時間再完成趨勢題；不要犧牲最後的乾淨重跑。


## 9. 我的解讀與驗收

請編輯這格，填自己的答案：
- 我的數據解讀（至少一個數字及單位）：……
- 我做的兩個驗證：……
- 若把 2025 改成 2024，我要改哪裡：……
- 還不確定的一件事：……

保存後重啟 kernel，從頭全跑。打開本次產出的 PNG，看標籤是否完整、是否與表一致。
合成資料只能練習分析，不能據此推論真實商店營運原因。